# Manga Diffusion — Evaluation

Generates 100 images per condition (baseline / blip2 / wd14) and computes FID, CLIP, and DINOv2 metrics.

**Before running:**
1. Runtime → Change runtime type → GPU (T4 minimum, A100 preferred)
2. Set BLIP2_LORA_PATH and WD14_LORA_PATH to your Drive paths below

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError("GPU not detected! Go to Runtime → Change runtime type → T4 GPU.")
print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

## 1. Configuration

In [ ]:
# ── EDIT THESE PATHS ─────────────────────────────────────────────────────────
BLIP2_LORA_PATH = "/content/drive/MyDrive/manga-diffusion/training/outputs/blip2-rank32/blip2-rank32.safetensors"
WD14_LORA_PATH  = "/content/drive/MyDrive/manga-diffusion/training/outputs/wd14-rank32/wd14-rank32.safetensors"
DRIVE_TEST_SET  = "/content/drive/MyDrive/manga-diffusion/data/test_set"
DRIVE_RESULTS   = "/content/drive/MyDrive/manga-diffusion/evaluation/results"
# ─────────────────────────────────────────────────────────────────────────────

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
print("Drive mounted.")

## 3. Install dependencies

In [ ]:
%%bash
pip install -q diffusers transformers accelerate peft xformers accelerate
pip install -q pytorch-fid open-clip-torch
pip install -q torchao --upgrade  # needed by peft for LoRA loading
echo "Dependencies installed."

## 4. Set up directories

In [ ]:
import os, shutil
from pathlib import Path

# Copy test set locally for faster metric computation
local_test = Path("/content/test_set")
local_test.mkdir(exist_ok=True)
for f in Path(DRIVE_TEST_SET).glob("*.png"):
    shutil.copy2(f, local_test / f.name)
print(f"Test set: {len(list(local_test.glob('*.png')))} images")

# Output directories
for cond in ["baseline", "blip2", "wd14"]:
    Path(f"/content/results/{cond}/generated").mkdir(parents=True, exist_ok=True)

## 5. Generate images — Baseline (no LoRA)

In [ ]:
import torch
from diffusers import StableDiffusionXLPipeline
from pathlib import Path

EVAL_PROMPTS = [
    "gknoir style, heavy ink linework, close-up panel, detective in rain, tense atmosphere",
    "gknoir style, thin precise linework, wide establishing shot, empty city street, melancholic",
    "gknoir style, heavy ink linework, over-shoulder panel, figure in doorway, stark shadows",
    "gknoir style, bold ink, action panel, two figures confronting, high tension",
    "gknoir style, fine linework, interior scene, desk with lamp, quiet solitude",
    "gknoir style, heavy ink, extreme close-up, eyes in shadow, mysterious",
    "gknoir style, crosshatch shading, mid-shot, woman in coat, rainy street",
    "gknoir style, stark contrast, panel sequence feel, running figure, urgency",
    "gknoir style, screentone shading, thoughtful expression, window light, introspective",
    "gknoir style, deep blacks, architecture detail, alley at night, foreboding",
]
SEEDS = list(range(10))

def generate_for(pipe, condition, prompts=EVAL_PROMPTS, seeds=SEEDS):
    out_dir = Path(f"/content/results/{condition}/generated")
    total = len(prompts) * len(seeds)
    done = 0
    for p_idx, prompt in enumerate(prompts):
        for seed in seeds:
            fname = out_dir / f"p{p_idx:02d}_s{seed:02d}.png"
            if fname.exists():
                done += 1
                continue
            gen = torch.Generator(device="cuda").manual_seed(seed)
            img = pipe(prompt=prompt, num_inference_steps=30, guidance_scale=7.5,
                       generator=gen, height=768, width=768).images[0]
            img.save(fname)
            done += 1
            if done % 10 == 0:
                print(f"[{condition}] {done}/{total}")
    print(f"[{condition}] Done — {len(list(out_dir.glob('*.png')))} images")

pipe_base = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16, use_safetensors=True).to("cuda")
pipe_base.set_progress_bar_config(disable=True)

generate_for(pipe_base, "baseline")

## 6. Generate images — BLIP-2 LoRA

In [ ]:
from pathlib import Path

# Unload base, load with BLIP-2 LoRA
pipe_blip2 = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16, use_safetensors=True).to("cuda")
pipe_blip2.set_progress_bar_config(disable=True)

lora_dir = str(Path(BLIP2_LORA_PATH).parent)
lora_name = Path(BLIP2_LORA_PATH).name
pipe_blip2.load_lora_weights(lora_dir, weight_name=lora_name)
print(f"Loaded LoRA: {lora_name}")

generate_for(pipe_blip2, "blip2")
del pipe_blip2
torch.cuda.empty_cache()

## 7. Generate images — WD14 LoRA

In [ ]:
pipe_wd14 = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16, use_safetensors=True).to("cuda")
pipe_wd14.set_progress_bar_config(disable=True)

lora_dir = str(Path(WD14_LORA_PATH).parent)
lora_name = Path(WD14_LORA_PATH).name
pipe_wd14.load_lora_weights(lora_dir, weight_name=lora_name)
print(f"Loaded LoRA: {lora_name}")

generate_for(pipe_wd14, "wd14")
del pipe_wd14
torch.cuda.empty_cache()

## 8. Compute FID

In [ ]:
from pytorch_fid import fid_score
import torch

results = {}
for cond in ["baseline", "blip2", "wd14"]:
    gen_dir = f"/content/results/{cond}/generated"
    fid = fid_score.calculate_fid_given_paths(
        [gen_dir, str(local_test)],
        batch_size=50, device="cuda", dims=2048)
    results.setdefault(cond, {})["fid"] = round(fid, 4)
    print(f"[{cond}] FID = {fid:.4f}")

## 9. Compute CLIP score

In [ ]:
from transformers import CLIPModel, CLIPProcessor
from PIL import Image
import re, torch, torch.nn.functional as F
from pathlib import Path

def prompt_for(fname):
    m = re.match(r"p(\d+)_s\d+\.png", fname)
    return EVAL_PROMPTS[int(m.group(1))] if m else EVAL_PROMPTS[0]

clip_model = CLIPModel.from_pretrained("openai/clip-vit-large-patch14").to("cuda")
clip_proc  = CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14")
clip_model.eval()

for cond in ["baseline", "blip2", "wd14"]:
    imgs = sorted(Path(f"/content/results/{cond}/generated").glob("*.png"))
    scores = []
    for img_path in imgs:
        prompt = prompt_for(img_path.name)
        image  = Image.open(img_path).convert("RGB")
        inputs = clip_proc(text=[prompt], images=image, return_tensors="pt",
                           padding=True, truncation=True).to("cuda")
        with torch.no_grad():
            out = clip_model(**inputs)
        scores.append(F.cosine_similarity(out.image_embeds, out.text_embeds).item())
    mean_clip = sum(scores) / len(scores)
    results.setdefault(cond, {})["clip_score"] = round(mean_clip, 4)
    print(f"[{cond}] CLIP score = {mean_clip:.4f}")

del clip_model
torch.cuda.empty_cache()

## 10. Compute DINOv2 style similarity

In [ ]:
from transformers import AutoModel, AutoProcessor
from PIL import Image
import torch, torch.nn.functional as F
from pathlib import Path

dino_model = AutoModel.from_pretrained("facebook/dinov2-large").to("cuda")
dino_proc  = AutoProcessor.from_pretrained("facebook/dinov2-large")
dino_model.eval()

def extract_feats(paths, desc):
    feats = []
    for i, p in enumerate(paths):
        img = Image.open(p).convert("RGB")
        inp = dino_proc(images=img, return_tensors="pt").to("cuda")
        with torch.no_grad():
            out = dino_model(**inp)
        feats.append(out.last_hidden_state[:, 0, :].cpu())
        if (i+1) % 20 == 0:
            print(f"  {desc} {i+1}/{len(paths)}")
    return torch.cat(feats, dim=0)

test_imgs  = sorted(local_test.glob("*.png"))
test_feats = extract_feats(test_imgs, "test")
test_norm  = F.normalize(test_feats, dim=1)

for cond in ["baseline", "blip2", "wd14"]:
    gen_imgs  = sorted(Path(f"/content/results/{cond}/generated").glob("*.png"))
    gen_feats = extract_feats(gen_imgs, cond)
    gen_norm  = F.normalize(gen_feats, dim=1)
    sim = (gen_norm @ test_norm.T).mean().item()
    results.setdefault(cond, {})["dino_sim"] = round(sim, 4)
    print(f"[{cond}] DINOv2 style sim = {sim:.4f}")

del dino_model
torch.cuda.empty_cache()

## 11. Save results CSV to Drive

In [ ]:
import csv, os
from pathlib import Path

# Print summary
print("
=== RESULTS SUMMARY ===")
print(f"{'Condition':<12} {'FID':>8} {'CLIP':>8} {'DINO':>8}")
print("-" * 42)
for cond in ["baseline", "blip2", "wd14"]:
    r = results.get(cond, {})
    print(f"{cond:<12} {r.get('fid','--'):>8} {r.get('clip_score','--'):>8} {r.get('dino_sim','--'):>8}")

# Save to Drive
drive_results = Path(DRIVE_RESULTS)
drive_results.mkdir(parents=True, exist_ok=True)
csv_path = drive_results / "metrics_summary.csv"
with open(csv_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["run_name", "fid", "clip_score", "dino_sim", "notes"])
    writer.writeheader()
    for cond in ["baseline", "blip2", "wd14"]:
        r = results.get(cond, {})
        writer.writerow({"run_name": cond, "fid": r.get("fid",""),
                         "clip_score": r.get("clip_score",""),
                         "dino_sim": r.get("dino_sim",""), "notes": ""})
print(f"CSV saved to {csv_path}")

# Copy all generated images to Drive
for cond in ["baseline", "blip2", "wd14"]:
    dst = drive_results / cond / "generated"
    dst.mkdir(parents=True, exist_ok=True)
    src = Path(f"/content/results/{cond}/generated")
    for img in src.glob("*.png"):
        shutil.copy2(img, dst / img.name)
    print(f"Copied {len(list(src.glob('*.png')))} images → {dst}")

## 12. Quick qualitative preview

In [ ]:
from IPython.display import display
from PIL import Image
import numpy as np
from pathlib import Path

# Show same prompt, same seed across 3 conditions
prompt_idx, seed = 0, 0
fname = f"p{prompt_idx:02d}_s{seed:02d}.png"
row = []
for cond in ["baseline", "blip2", "wd14"]:
    p = Path(f"/content/results/{cond}/generated/{fname}")
    row.append(np.array(Image.open(p)))

grid = np.concatenate(row, axis=1)
img_grid = Image.fromarray(grid)
img_grid = img_grid.resize((img_grid.width // 2, img_grid.height // 2))
display(img_grid)
print(f"Prompt: {EVAL_PROMPTS[prompt_idx]}")
print("Left: baseline | Middle: blip2 | Right: wd14")